# Upgraded Spiking Conv NN Training (SCNN)
This notebook uses the upgraded 3-block STDP SCNN from `snn.py` on the APTOS retinal dataset.
It applies optional class balancing before spike encoding, trains with stratified STDP scheduling, and evaluates balanced-SVC plus prototypical few-shot readouts.
It also saves spike timing/frequency/pattern metrics, split-wise PRF metrics, confusion matrices, and model checkpoints in the outputs directory.

In [1]:
# Install required packages (run once)
%pip install -q numpy scipy tqdm scikit-learn matplotlib pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import time
import importlib
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    )
import matplotlib.pyplot as plt

def resolve_scnn_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Retinal Classification" / "SCNN",
        Path.cwd() / "SCNN",
        Path.cwd().parent / "SCNN",
    ]
    for candidate in candidates:
        if (candidate / "snn.py").exists() and (candidate / "utils.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate SCNN directory containing snn.py and utils.py.")

SCNN_DIR = resolve_scnn_dir()
if str(SCNN_DIR) not in sys.path:
    sys.path.insert(0, str(SCNN_DIR))

import utils as scnn_utils
import snn

# Force reload so notebook always sees latest local edits to utils.py and snn.py.
scnn_utils = importlib.reload(scnn_utils)
snn = importlib.reload(snn)

print("SCNN directory:", SCNN_DIR)
print("utils module file:", scnn_utils.__file__)
print("snn module file:", snn.__file__)
print("load_encoded_retinal_dataset signature:", inspect.signature(scnn_utils.load_encoded_retinal_dataset))
print("Torch version:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
else:
    print("CUDA unavailable. Notebook will still train on CPU.")

SCNN directory: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN
utils module file: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/utils.py
snn module file: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/snn.py
load_encoded_retinal_dataset signature: (data_prop=1, nb_timesteps=15, image_size=64, threshold=15, filters=None, seed=0, oversample=True, oversample_strategy='oversample', augment_minority=True)
Torch version: 2.10.0+cu128
Torch CUDA build: 12.8
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4070 Laptop GPU


In [3]:
DATASET_ROOT = scnn_utils.resolve_retinal_dataset_root()
IMAGE_SIZE = 64

print("Retinal dataset root:", DATASET_ROOT)
print("Retinal image size:", IMAGE_SIZE)

Retinal dataset root: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/dataset/aptos2019-blindness-detection/imagefolder
Retinal image size: 64


In [4]:
# Optional patch: run conv/pool torch ops on CUDA when available.
from torch.nn.functional import conv2d, max_pool2d

def patch_scnn_for_device(scnn_module, device):
    def pool_call(self, in_spks):
        in_spks = np.pad(
            in_spks,
            ((0, 0), (self.padding[0], self.padding[0]), (self.padding[1], self.padding[1])),
            mode="constant",
        )
        in_spks_t = torch.as_tensor(in_spks, dtype=torch.float32, device=device).unsqueeze(0)
        out_spks = max_pool2d(in_spks_t, self.kernel_size, stride=self.stride).detach().cpu().numpy()[0]
        out_spks = out_spks * self.active_neurons
        self.active_neurons[out_spks == 1] = False
        return out_spks

    def conv_get_conv_of(self, input_spks, output_neuron):
        _, n_h, n_w = output_neuron
        input_t = torch.as_tensor(input_spks, dtype=torch.float32, device=device).unsqueeze(0)
        convs = torch.nn.functional.unfold(
            input_t,
            kernel_size=self.kernel_size,
            stride=self.stride,
        )[0].detach().cpu().numpy()
        conv_ind = (n_h * self.pot.shape[2]) + n_w
        return convs[:, conv_ind]

    def conv_call(self, spk_in, train=False):
        spk_in = np.pad(
            spk_in,
            ((0, 0), (self.padding[0], self.padding[0]), (self.padding[1], self.padding[1])),
            mode="constant",
        )

        self.recorded_spks += spk_in
        spk_out = np.zeros(self.pot.shape)

        x = torch.as_tensor(spk_in, dtype=torch.float32, device=device).unsqueeze(0)
        weights = torch.as_tensor(self.weights, dtype=torch.float32, device=device)
        out_conv = conv2d(x, weights, stride=self.stride).detach().cpu().numpy()[0]

        self.pot[self.active_neurons] += out_conv[self.active_neurons]
        output_spikes = self.pot > self.firing_threshold

        if np.any(output_spikes):
            spk_out[output_spikes] = 1
            spk_out = self.lateral_inhibition(spk_out)

            if train and self.plasticity:
                winners = self.get_winners()
                for winner in winners:
                    self.stdp(winner)

            self.pot[spk_out == 1] = self.v_reset
            self.active_neurons[spk_out == 1] = False

        return spk_out

    scnn_module.SpikingPool.__call__ = pool_call
    scnn_module.SpikingConv.get_conv_of = conv_get_conv_of
    scnn_module.SpikingConv.__call__ = conv_call

patch_scnn_for_device(snn, DEVICE)
print("SCNN patched for torch ops on:", DEVICE)

SCNN patched for torch ops on: cuda


In [5]:
import sys
import inspect
import importlib
from pathlib import Path

import numpy as np
import torch

def resolve_scnn_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Retinal Classification" / "SCNN",
        Path.cwd() / "SCNN",
        Path.cwd().parent / "SCNN",
    ]
    for candidate in candidates:
        if (candidate / "snn.py").exists() and (candidate / "utils.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate SCNN directory containing snn.py and utils.py.")

SCNN_DIR = resolve_scnn_dir()
if str(SCNN_DIR) not in sys.path:
    sys.path.insert(0, str(SCNN_DIR))

import utils as scnn_utils
import snn

# Ensure this cell uses the latest local edits even if earlier import cells were not rerun.
scnn_utils = importlib.reload(scnn_utils)
snn = importlib.reload(snn)

# Experiment config
SEED = 1
DATA_PROP = 0.2             # Use 1.0 for the full retinal train split
NB_TIMESTEPS = 15
IMAGE_SIZE = 64
EPOCHS_PER_LAYER = [5, 5, 3]
CONVERGENCE_THRESHOLD = 0.0  # 0.0 disables early stop by convergence

OVERSAMPLE = True
OVERSAMPLE_STRATEGY = "oversample"  # "oversample" or "hybrid"
AUGMENT_MINORITY = True
K_SHOT = 5

np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_DIR = SCNN_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("SCNN directory:", SCNN_DIR)
print("utils module file:", scnn_utils.__file__)
print("snn module file:", snn.__file__)
print("Output directory:", OUTPUT_DIR)

loader_signature = inspect.signature(scnn_utils.load_encoded_retinal_dataset)
loader_params = loader_signature.parameters
print("Runtime loader signature:", loader_signature)

load_kwargs = {
    "data_prop": DATA_PROP,
    "nb_timesteps": NB_TIMESTEPS,
    "image_size": IMAGE_SIZE,
    "seed": SEED,
}

if "oversample" in loader_params:
    load_kwargs["oversample"] = OVERSAMPLE
if "oversample_strategy" in loader_params:
    load_kwargs["oversample_strategy"] = OVERSAMPLE_STRATEGY
if "augment_minority" in loader_params:
    load_kwargs["augment_minority"] = AUGMENT_MINORITY

missing_keys = [k for k in ["oversample", "oversample_strategy", "augment_minority"] if k not in loader_params]
if missing_keys:
    print(f"Warning: runtime utils.py does not expose parameters: {missing_keys}. Running with supported args only.")

X_train, y_train, X_eval, y_eval = scnn_utils.load_encoded_retinal_dataset(**load_kwargs)

def stratified_split_indices(labels, test_ratio=0.5, seed=1):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    split_a_idx = []
    split_b_idx = []

    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        rng.shuffle(cls_idx)

        if len(cls_idx) <= 1:
            n_b = 1
        else:
            n_b = int(round(len(cls_idx) * test_ratio))
            n_b = max(1, n_b)
            n_b = min(n_b, len(cls_idx) - 1)

        split_b_idx.extend(cls_idx[:n_b].tolist())
        split_a_idx.extend(cls_idx[n_b:].tolist())

    rng.shuffle(split_a_idx)
    rng.shuffle(split_b_idx)
    return np.array(split_a_idx), np.array(split_b_idx)

val_idx, test_idx = stratified_split_indices(y_eval, test_ratio=0.5, seed=SEED)
X_val, y_val = X_eval[val_idx], y_eval[val_idx]
X_test, y_test = X_eval[test_idx], y_eval[test_idx]

input_shape = X_train[0][0].shape
model = snn.SNN(input_shape)

def print_distribution(name, labels):
    unique, counts = np.unique(labels, return_counts=True)
    pairs = {int(k): int(v) for k, v in zip(unique, counts)}
    print(f"{name} class distribution: {pairs}")

print(f"Input shape: {X_train[0].shape} ({np.prod(X_train[0].shape)} values)")
print(f"Output shape: {model.output_shape} ({np.prod(model.output_shape)} values)")
print(f"Mean spikes count per input: {X_train.mean(0).sum()}")
print(f"Train samples: {len(X_train)} | Val samples: {len(X_val)} | Test samples: {len(X_test)}")
print(f"Trainable conv layers: {model.nb_trainable_layers}")
print("Epoch schedule:", EPOCHS_PER_LAYER)
print_distribution("Train", y_train)
print_distribution("Val", y_val)
print_distribution("Test", y_test)

SCNN directory: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN
utils module file: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/utils.py
snn module file: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/snn.py
Output directory: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs
Runtime loader signature: (data_prop=1, nb_timesteps=15, image_size=64, threshold=15, filters=None, seed=0, oversample=True, oversample_strategy='oversample', augment_minority=True)
[utils] Class distribution before balancing: [(0, 279), (1, 55), (2, 166), (3, 34), (4, 51)]
[utils] Class distribution after  balancing: [(0, 279), (1, 279), (2, 279), (3, 279), (4, 279)]
[utils] Encoding train spikes …
[utils] Encoding test  spikes …
Input shape: (15, 2, 64, 64) (122880 values)
Output shape: (128, 8, 8) (8192 values)
Mean spikes count per input: 1293.2960573476703
Train samples: 1395 | Val samples: 366 | Test samples: 367
Trainable conv laye

In [6]:
print("\n### TRAINING ###")
train_start = time.time()
training_log_rows = []

for layer in range(model.nb_trainable_layers):
    n_epochs = EPOCHS_PER_LAYER[layer] if layer < len(EPOCHS_PER_LAYER) else EPOCHS_PER_LAYER[-1]
    print(f"Layer {layer + 1}/{model.nb_trainable_layers}...")

    for epoch in range(1, n_epochs + 1):
        epoch_start = time.time()
        rng = np.random.default_rng(SEED + layer * 1000 + epoch)
        indices = snn._stratified_shuffle(y_train, rng)
        epoch_iter = tqdm(indices, total=len(indices), leave=False)
        conv_scores = []

        for i in epoch_iter:
            model(X_train[i], train_layer=layer)
            conv_score = float(model.conv_layers[layer].get_learning_convergence())
            conv_scores.append(conv_score)
            epoch_iter.set_postfix(convergence=f"{conv_score:.6f}")

        mean_conv = float(np.mean(conv_scores)) if conv_scores else float("nan")
        last_conv = float(conv_scores[-1]) if conv_scores else float("nan")
        epoch_seconds = float(time.time() - epoch_start)

        training_log_rows.append({
            "layer": int(layer + 1),
            "epoch": int(epoch),
            "mean_learning_convergence": mean_conv,
            "last_learning_convergence": last_conv,
            "epoch_seconds": epoch_seconds,
            "samples_seen": int(len(indices)),
        })

        print(
            f"  Epoch {epoch}/{n_epochs} | "
            f"Mean convergence: {mean_conv:.6f} | "
            f"Last convergence: {last_conv:.6f} | "
            f"Time: {epoch_seconds:.2f}s"
        )

        if CONVERGENCE_THRESHOLD > 0 and last_conv < CONVERGENCE_THRESHOLD:
            print(
                f"  Early stop for layer {layer + 1} at epoch {epoch} "
                f"(convergence {last_conv:.6f} < {CONVERGENCE_THRESHOLD})"
            )
            break

train_elapsed = time.time() - train_start
print(f"Training finished in {train_elapsed/60:.2f} min")

training_log_df = pd.DataFrame(training_log_rows)
training_log_path = OUTPUT_DIR / "scnn_training_log.csv"
training_log_df.to_csv(training_log_path, index=False)
print("Saved training log to:", training_log_path)

checkpoint_path = OUTPUT_DIR / "scnn_model_state.pth"
torch.save(
    {
        "seed": SEED,
        "nb_timesteps": NB_TIMESTEPS,
        "image_size": IMAGE_SIZE,
        "output_shape": model.output_shape,
        "epochs_per_layer": EPOCHS_PER_LAYER,
        "convergence_threshold": CONVERGENCE_THRESHOLD,
        "oversample": OVERSAMPLE,
        "oversample_strategy": OVERSAMPLE_STRATEGY,
        "augment_minority": AUGMENT_MINORITY,
        "conv_layers": [
            {
                "weights": layer_obj.weights,
                "firing_threshold": layer_obj.firing_threshold,
                "stdp_count": layer_obj.stdp_cnt,
            }
            for layer_obj in model.conv_layers
        ],
    },
    checkpoint_path,
 )
print("Saved SCNN model checkpoint to:", checkpoint_path)


### TRAINING ###
Layer 1/3...


  Epoch 1/5 | Mean convergence: 0.034979 | Last convergence: 0.001335 | Time: 223.06s


  Epoch 2/5 | Mean convergence: 0.000667 | Last convergence: 0.000298 | Time: 269.23s


  Epoch 3/5 | Mean convergence: 0.000184 | Last convergence: 0.000219 | Time: 263.49s


  Epoch 4/5 | Mean convergence: 0.000263 | Last convergence: 0.000159 | Time: 238.58s


...

  Epoch 4/5 | Mean convergence: 0.000013 | Last convergence: 0.000023 | Time: 795.51s


  Epoch 5/5 | Mean convergence: 0.000012 | Last convergence: 0.000005 | Time: 716.24s
Layer 3/3...


  Epoch 1/3 | Mean convergence: 0.044012 | Last convergence: 0.000192 | Time: 810.50s


  Epoch 2/3 | Mean convergence: 0.000076 | Last convergence: 0.000024 | Time: 1006.55s


  Epoch 3/3 | Mean convergence: -0.000000 | Last convergence: 0.000005 | Time: 1166.34s
Training finished in 121.53 min
Saved training log to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/scnn_training_log.csv
Saved SCNN model checkpoint to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/scnn_model_state.pth


In [7]:
print("\n### FEATURE EXTRACTION ###")

def compute_spike_metrics(spikes):
    # spikes shape: (samples, timesteps, channels, height, width)
    binary_spikes = (spikes > 0).astype(np.uint8)
    samples, timesteps = binary_spikes.shape[0], binary_spikes.shape[1]
    neurons = int(np.prod(binary_spikes.shape[2:]))

    flat_spikes = binary_spikes.reshape(samples, timesteps, -1)
    active_mask = flat_spikes.any(axis=2)

    first_spike_t = np.argmax(active_mask, axis=1).astype(np.float32)
    no_spike_mask = ~active_mask.any(axis=1)
    first_spike_t[no_spike_mask] = np.nan

    total_spikes = flat_spikes.sum(axis=(1, 2)).astype(np.float32)
    spike_frequency = total_spikes / float(timesteps)
    spike_rate = total_spikes / float(timesteps * neurons)
    active_neuron_fraction = flat_spikes.any(axis=1).sum(axis=1) / float(neurons)

    sample_metrics = pd.DataFrame({
        "first_spike_timestep": first_spike_t,
        "spike_frequency": spike_frequency,
        "spike_rate": spike_rate,
        "active_neuron_fraction": active_neuron_fraction,
        "total_spikes": total_spikes,
    })

    summary_metrics = {
        "first_spike_timestep_mean": float(np.nanmean(first_spike_t)) if np.isfinite(first_spike_t).any() else np.nan,
        "first_spike_timestep_std": float(np.nanstd(first_spike_t)) if np.isfinite(first_spike_t).any() else np.nan,
        "spike_frequency_mean": float(np.mean(spike_frequency)),
        "spike_frequency_std": float(np.std(spike_frequency)),
        "spike_rate_mean": float(np.mean(spike_rate)),
        "spike_rate_std": float(np.std(spike_rate)),
        "active_neuron_fraction_mean": float(np.mean(active_neuron_fraction)),
        "active_neuron_fraction_std": float(np.std(active_neuron_fraction)),
        "pattern_sparsity_mean": float(1.0 - np.mean(spike_rate)),
        "total_spikes_mean": float(np.mean(total_spikes)),
    }

    return sample_metrics, summary_metrics

def collect_split_features(split_name, X_split):
    split_spikes = []
    split_features = []

    for x in tqdm(X_split, desc=f"Extract {split_name}"):
        spk = model(x)
        split_spikes.append(spk)
        split_features.append(snn.extract_features(spk))

    split_spikes = np.stack(split_spikes).astype(np.float32)
    split_features = np.asarray(split_features, dtype=np.float32)

    sample_metrics_df, summary_metrics = compute_spike_metrics(split_spikes)
    sample_metrics_path = OUTPUT_DIR / f"{split_name}_spike_sample_metrics.csv"
    sample_metrics_df.to_csv(sample_metrics_path, index=False)
    print(f"Saved {split_name} spike sample metrics to:", sample_metrics_path)
    print(f"{split_name} feature matrix shape: {split_features.shape}")

    return split_features, summary_metrics

F_train, train_spike_summary = collect_split_features("train", X_train)
F_val, val_spike_summary = collect_split_features("val", X_val)
F_test, test_spike_summary = collect_split_features("test", X_test)

spike_summary_rows = []
for split_name, summary in [
    ("train", train_spike_summary),
    ("val", val_spike_summary),
    ("test", test_spike_summary),
]:
    row = {"split": split_name}
    row.update(summary)
    spike_summary_rows.append(row)

spike_summary_df = pd.DataFrame(spike_summary_rows)
spike_summary_path = OUTPUT_DIR / "scnn_spike_summary_metrics.csv"
spike_summary_df.to_csv(spike_summary_path, index=False)
print("Saved spike summary metrics to:", spike_summary_path)

print(f"Mean total number of spikes per sample: {np.mean(model.recorded_sum_spks):.4f}")


### FEATURE EXTRACTION ###


Extract train: 100%|██████████| 1395/1395 [19:48<00:00,  1.17it/s]


Saved train spike sample metrics to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/train_spike_sample_metrics.csv
train feature matrix shape: (1395, 32768)


Extract val: 100%|██████████| 366/366 [05:08<00:00,  1.19it/s]


Saved val spike sample metrics to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/val_spike_sample_metrics.csv
val feature matrix shape: (366, 32768)


Extract test: 100%|██████████| 367/367 [05:14<00:00,  1.17it/s]

Saved test spike sample metrics to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/test_spike_sample_metrics.csv
test feature matrix shape: (367, 32768)
Saved spike summary metrics to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/scnn_spike_summary_metrics.csv
Mean total number of spikes per sample: 3349.4507


In [8]:
def evaluate_split_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )
    return float(acc), float(precision), float(recall), float(f1)

def save_split_artifacts(method_name, split_name, y_train_ref, y_true, y_pred):
    labels_eval = sorted(np.unique(np.concatenate([y_train_ref, y_true])).tolist())
    target_names = [str(label) for label in labels_eval]

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=labels_eval,
            target_names=target_names,
            output_dict=True,
            zero_division=0,
        )
    ).transpose()
    report_path = OUTPUT_DIR / f"{method_name}_{split_name}_classification_report.csv"
    report_df.to_csv(report_path)

    cm = confusion_matrix(y_true, y_pred, labels=labels_eval)
    cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
    cm_path = OUTPUT_DIR / f"{method_name}_{split_name}_confusion_matrix.csv"
    cm_df.to_csv(cm_path)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(labels_eval)),
        yticks=np.arange(len(labels_eval)),
        xticklabels=target_names,
        yticklabels=target_names,
        ylabel="True label",
        xlabel="Predicted label",
        title=f"Confusion Matrix - {method_name} - {split_name}",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    threshold = cm.max() / 2.0 if cm.size else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > threshold else "black",
            )

    plt.tight_layout()
    cm_img_path = OUTPUT_DIR / f"{method_name}_{split_name}_confusion_matrix.png"
    plt.savefig(cm_img_path, dpi=200)
    plt.close(fig)

    print("  Saved report to:", report_path)
    print("  Saved confusion matrix CSV to:", cm_path)
    print("  Saved confusion matrix image to:", cm_img_path)

def evaluate_readout_method(method_name, predictor, split_features_map, split_labels_map, y_train_ref):
    metrics_rows = []
    for split_name, features in split_features_map.items():
        y_true = split_labels_map[split_name]
        y_pred = predictor(features)

        acc, precision, recall, f1 = evaluate_split_predictions(y_true, y_pred)
        metrics_rows.append({
            "method": method_name,
            "split": split_name,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
        })

        print(f"{method_name} | {split_name} -> Acc: {acc:.4f}, P: {precision:.4f}, R: {recall:.4f}, F1: {f1:.4f}")
        save_split_artifacts(method_name, split_name, y_train_ref, y_true, y_pred)

    return metrics_rows

split_features = {
    "train": F_train,
    "val": F_val,
    "test": F_test,
}
split_labels = {
    "train": y_train,
    "val": y_val,
    "test": y_test,
}

all_metrics = []

# Method 1: Balanced LinearSVC readout
balanced_svc = snn.train_balanced_svc(F_train, y_train, seed=SEED)
all_metrics.extend(
    evaluate_readout_method(
        "balanced_svc",
        lambda feats: balanced_svc.predict(feats),
        split_features,
        split_labels,
        y_train,
    )
 )

# Method 2: Prototypical readout using full train support
proto_full = snn.PrototypicalReadout().fit(F_train, y_train)
all_metrics.extend(
    evaluate_readout_method(
        "proto_full",
        lambda feats: proto_full.predict(feats),
        split_features,
        split_labels,
        y_train,
    )
 )

# Method 3: Prototypical readout in few-shot mode
proto_kshot = snn.PrototypicalReadout().fit_k_shot(
    X_train, y_train, model, k_shot=K_SHOT, seed=SEED
 )
all_metrics.extend(
    evaluate_readout_method(
        f"proto_{K_SHOT}shot",
        lambda feats: proto_kshot.predict(feats),
        split_features,
        split_labels,
        y_train,
    )
 )

metrics_df = pd.DataFrame(all_metrics)
metrics_path = OUTPUT_DIR / "scnn_readout_split_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("\nSaved split metrics to:", metrics_path)

summary_view = metrics_df.sort_values(["method", "split"]).reset_index(drop=True)
summary_view

/media/aejaz/New Volume/Projects/SNN/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


balanced_svc | train -> Acc: 0.9986, P: 0.9986, R: 0.9986, F1: 0.9986
  Saved report to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/balanced_svc_train_classification_report.csv
  Saved confusion matrix CSV to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/balanced_svc_train_confusion_matrix.csv
  Saved confusion matrix image to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/balanced_svc_train_confusion_matrix.png
balanced_svc | val -> Acc: 0.6967, P: 0.6914, R: 0.6967, F1: 0.6935
  Saved report to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/balanced_svc_val_classification_report.csv
  Saved confusion matrix CSV to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/balanced_svc_val_confusion_matrix.csv
  Saved confusion matrix image to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/balanced_svc_val_confusion_matrix.png
balanc

Prototypes (5-shot): 100%|██████████| 25/25 [00:20<00:00,  1.20it/s]


proto_5shot | train -> Acc: 0.3075, P: 0.3893, R: 0.3075, F1: 0.3206
  Saved report to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/proto_5shot_train_classification_report.csv
  Saved confusion matrix CSV to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/proto_5shot_train_confusion_matrix.csv
  Saved confusion matrix image to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/proto_5shot_train_confusion_matrix.png
proto_5shot | val -> Acc: 0.4016, P: 0.5764, R: 0.4016, F1: 0.4451
  Saved report to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/proto_5shot_val_classification_report.csv
  Saved confusion matrix CSV to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/proto_5shot_val_confusion_matrix.csv
  Saved confusion matrix image to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/SCNN/outputs/proto_5shot_val_confusion_matrix.png
proto_5shot | 

,method,split,accuracy,precision,recall,f1_score
0,balanced_svc,test,0.651226,0.632158,0.651226,0.640625
1,balanced_svc,train,0.998566,0.998569,0.998566,0.998566
2,balanced_svc,val,0.696721,0.691405,0.696721,0.693467
3,proto_5shot,test,0.365123,0.547942,0.365123,0.402190
4,proto_5shot,train,0.307527,0.389318,0.307527,0.320610
5,proto_5shot,val,0.401639,0.576398,0.401639,0.445111
6,proto_full,test,0.604905,0.583947,0.604905,0.582533
7,proto_full,train,0.545520,0.643546,0.545520,0.547815
8,proto_full,val,0.647541,0.641793,0.647541,0.620092
